# DSA 8301 — Statistical Inference for Big Data
## Kenya Housing Survey 2023/24 — Data Loading, Understanding & Exploration

**Student:** Valerie Jerono | **Reg No:** [Your Reg No]  
**Course:** DSA 8301 — Statistical Inference for Big Data  
**Lecturer:** Prof. Jacob Ong'ala  
**Institution:** Strathmore University  
**Date:** June 2026  

---

### Dataset Source
Kenya National Bureau of Statistics (KNBS) — *Kenya Housing Survey 2023/24*  
Portal: https://statistics.knbs.or.ke/nada/index.php/catalog/184/get-microdata

---

> **Scope of this notebook:** Data loading → variable inventory → preprocessing → descriptive statistics → graphical EDA → distributional assessment.  
> Parametric and non-parametric inference follow in a separate notebook.


---
## 0. Environment Setup

In [ ]:
# ── 0.1  Mount Google Drive ──────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
# ── 0.2  Install dependencies (first run only) ───────────────────────
!pip install -q pyreadstat polars pyarrow
print('Dependencies ready.')

In [ ]:
# ── 0.3  Core imports ────────────────────────────────────────────────
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.stats import shapiro, probplot, norm as spnorm
from pathlib import Path

warnings.filterwarnings('ignore')
np.random.seed(42)

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 40)

plt.rcParams.update({
    'figure.dpi': 130, 'figure.facecolor': 'white',
    'axes.facecolor': '#F8F8F6', 'axes.spines.top': False,
    'axes.spines.right': False, 'axes.titlesize': 13,
    'axes.titleweight': '600', 'axes.labelsize': 11,
    'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'font.family': 'sans-serif', 'legend.fontsize': 9,
})

TEAL   = '#00695C'; RED    = '#B71C1C'; AMBER  = '#E65100'
BLUE   = '#1565C0'; PURPLE = '#6A1B9A'; GRAY   = '#546E7A'
DARK   = '#2C2C2A'; GREEN  = '#2E7D32'

print('All imports loaded.')

In [ ]:
# ── 0.4  Paths and county map ────────────────────────────────────────
DRIVE = Path('/content/drive/MyDrive/KHS_Dissertation')
PQ    = DRIVE / 'data' / 'parquet'
RAW   = DRIVE / 'data' / 'raw'
FIGS  = DRIVE / 'outputs' / 'figures' / 'dsa8301'
TABS  = DRIVE / 'outputs' / 'tables'  / 'dsa8301'
for p in [FIGS, TABS]: p.mkdir(parents=True, exist_ok=True)

COUNTY_MAP = {
     1:'Mombasa',        2:'Kwale',          3:'Kilifi',         4:'Tana River',
     5:'Lamu',           6:'Taita-Taveta',   7:'Garissa',        8:'Wajir',
     9:'Mandera',       10:'Marsabit',      11:'Isiolo',        12:'Meru',
    13:'Tharaka-Nithi', 14:'Embu',          15:'Kitui',         16:'Machakos',
    17:'Makueni',       18:'Nyandarua',     19:'Nyeri',         20:'Kirinyaga',
    21:"Murang'a",      22:'Kiambu',        23:'Turkana',       24:'West Pokot',
    25:'Samburu',       26:'Trans Nzoia',   27:'Uasin Gishu',   28:'Elgeyo-Marakwet',
    29:'Nandi',         30:'Baringo',       31:'Laikipia',      32:'Nakuru',
    33:'Narok',         34:'Kajiado',       35:'Kericho',       36:'Bomet',
    37:'Kakamega',      38:'Vihiga',        39:'Bungoma',       40:'Busia',
    41:'Siaya',         42:'Kisumu',        43:'Homa Bay',      44:'Migori',
    45:'Kisii',         46:'Nyamira',       47:'Nairobi',
}
print(f'Paths ready.  FIGS={FIGS}  TABS={TABS}')

---
## 1. Dataset Description

### 1.1 Source & Background

The **Kenya Housing Survey (KHS) 2023/24** is a nationally representative household survey conducted by KNBS. It covers **21,347 households** across all **47 counties**.

The survey is distributed as Stata (.dta) files, stored as Parquet here. Three files join directly to the household spine on `interview__key`. Land parcels are structural — only households that own land (i00 == 1, ~45.5%) have parcel records.

| File key | Unit | Rows | Core content | Join key |
|---|---|---|---|---|
| `household` | Household (spine) | 21,347 | Finances, tenure, utilities, 392 cols | — |
| `dwelling` | Dwelling unit | 25,116 | Wall/roof/floor, rooms | `interview__key` |
| `individual` | Person | 80,889 | Demographics, education | `interview__key` |
| `land_parcels` | Land parcel | 11,136 | Tenure, title docs, eviction risk | `interview__key` (owners only) |

In [ ]:
# ── 1.2  Load parquet files ──────────────────────────────────────────
# Adjust parquet filenames to match what you have on Drive.
# The keys below are used throughout this notebook.

FILE_MAP = {
    'household'   : 'Household_Information_Data.parquet',
    'individual'  : 'Individual_Data.parquet',
    'dwelling'    : 'Dwelling_Units_Data.parquet',
    'land_parcels': 'Land_Parcels_Data.parquet',
}

dfs = {}
print(f'  {"File":<15} {"Rows":>8}  {"Cols":>6}  {"Key present?":<15}')
print('  ' + '-'*52)
for key, fname in FILE_MAP.items():
    path = PQ / fname
    if not path.exists():
        print(f'  {key:<15}  NOT FOUND at {path}')
        continue
    df = pd.read_parquet(path)
    dfs[key] = df
    has_key = 'interview__key' in df.columns
    print(f'  {key:<15} {df.shape[0]:>8,}  {df.shape[1]:>6}  {str(has_key):<15}')

hh  = dfs['household']
ind = dfs['individual']
dw  = dfs['dwelling']
lp  = dfs['land_parcels']
print(f'\nHousehold spine: {hh.shape[0]:,} rows x {hh.shape[1]:,} cols')

In [ ]:
# ── 1.3  Variable registry — all analysis variables ─────────────────
# These span the four required types: continuous, ordinal, binary, categorical.
# Column names verified against the KHS 2023/24 Stata codebook.
#
# Key fixes vs earlier drafts:
#   k02  → insecurity flag is k02 == 2 (No), NOT k02 == 0 (code 0 does not exist)
#   k25  → willingness-to-pay ceiling, NOT rent paid (use k05 for rent)
#   j09/j10/j11 → opinion questions — removed from physical hazard (D3)
#   e06/e07/e08 → correct D3 hazard variables

VARIABLE_REGISTRY = {
    # ── Continuous ────────────────────────────────────────────────────
    'k05' : {'label': 'Monthly rent paid (KES)',            'type': 'continuous',  'file': 'household',
             'notes': 'Structural miss: renters only (~32.5% of HHs). k05 < 0 impossible — flag negatives.'},
    'l14' : {'label': 'Estimated dwelling value (KES)',     'type': 'continuous',  'file': 'household',
             'notes': 'Structural miss: owner-occupiers only (~61.6% of HHs).'},
    'l15' : {'label': 'Imputed monthly housing cost (KES)', 'type': 'continuous',  'file': 'household',
             'notes': 'Structural miss: owner-occupiers only. Owner equivalent rent.'},
    # ── Ordinal ───────────────────────────────────────────────────────
    'c01_1': {'label': 'Main drinking water source',        'type': 'ordinal',     'file': 'household',
              'notes': '1=Public water co. … 10=Surface water. Lower = safer.'},
    'c04'  : {'label': 'Toilet facility type',             'type': 'ordinal',     'file': 'household',
              'notes': '1=Flush-piped sewer (best) … 13=No facility/bush (worst).'},
    'c10'  : {'label': 'Main electricity source',          'type': 'ordinal',     'file': 'household',
              'notes': '1=KPLC grid … 12=None.'},
    'c11'  : {'label': 'Main cooking fuel',                'type': 'ordinal',     'file': 'household',
              'notes': '9=Firewood … 13=Dung = worst.  7=LPG = clean.'},
    'e06'  : {'label': 'Flood exposure (0=none/1=severe/2=mild)',   'type': 'ordinal', 'file': 'household',
              'notes': 'Enumerator observation. Near-zero missingness.'},
    'e07'  : {'label': 'Mudslide exposure (0=none/1=severe/2=mild)','type': 'ordinal', 'file': 'household',
              'notes': 'Enumerator observation.'},
    'e08'  : {'label': 'Terrain type (1=flat … 4=steep)',  'type': 'ordinal',     'file': 'household',
              'notes': 'Enumerator observation.'},
    # ── Binary ────────────────────────────────────────────────────────
    'i00'  : {'label': 'Land ownership (0=No, 1=Yes)',      'type': 'binary',      'file': 'household',
              'notes': 'Near-zero miss. i00==0 IS the D2 insecurity flag for non-owners.'},
    'k02'  : {'label': 'Written tenancy agreement (1=Yes, 2=No)', 'type': 'binary', 'file': 'household',
              'notes': 'Structural miss: renters only (67.5% missing). Insecurity = k02 == 2.'},
    'a07_1': {'label': 'Urban/Rural (1=Urban, 2=Rural)',    'type': 'binary',      'file': 'household',
              'notes': 'Clean. Urban = 56% of sample.'},
    # ── Categorical ───────────────────────────────────────────────────
    'a01'  : {'label': 'County code (1–47)',                'type': 'categorical', 'file': 'household',
              'notes': '47 unique values. Map via COUNTY_MAP.'},
}

print(f'  {"Var":<8}  {"Type":<12}  {"Label"}')
print('  ' + '-'*68)
for v, info in VARIABLE_REGISTRY.items():
    print(f'  {v:<8}  {info["type"]:<12}  {info["label"]}')

type_counts = pd.Series([v['type'] for v in VARIABLE_REGISTRY.values()]).value_counts()
print(f'\nType summary: {dict(type_counts)}')
print('\nTotal variables in registry:', len(VARIABLE_REGISTRY))

---
## 2. Data Preprocessing

### 2.1 Building the Master Spine

All analysis merges onto the **household** file as the primary key.
Three merge steps are needed:

1. **Dwelling** → aggregate to one row per household (worst dwelling wins for vulnerability)
2. **Individual** → aggregate to household-level summaries (size, age structure)
3. **Land parcels** → aggregate to one row per household (only i00 == 1 HHs have parcels; non-owners receive worst-case imputation)

After merging, all analysis columns carry their original KHS variable names. No renaming except for computed columns, which are prefixed `hh_` or `lp_`.

In [ ]:
# ── 2.1a  Dwelling aggregation ───────────────────────────────────────
# Some HHs have 2-4 dwelling units. Take the WORST (most vulnerable)
# values to represent the household.

# Material quality scores (higher = better)
FLOOR_SCORE = {1:1, 2:1, 3:2, 4:1, 5:3, 6:3, 7:3, 8:3, 9:3, 96:2}  # d14
WALL_SCORE  = {1:1, 2:1, 3:1, 4:1, 5:1, 6:1, 7:1, 8:1, 9:1,         # d15
               10:2, 11:3, 12:3, 13:3, 14:3, 15:2, 16:2, 17:3, 96:2}
ROOF_SCORE  = {1:1, 2:1, 3:2, 4:1, 5:2, 6:3, 7:3, 8:1, 96:2}        # d16

dw2 = dw.copy()
dw2['floor_score'] = dw2['d14'].map(FLOOR_SCORE).fillna(2)
dw2['wall_score']  = dw2['d15'].map(WALL_SCORE ).fillna(2)
dw2['roof_score']  = dw2['d16'].map(ROOF_SCORE ).fillna(2)

dw_agg = (
    dw2.groupby('interview__key')
    .agg(
        d03_worst   = ('d03',  'max'),   # worst dwelling type (higher code = poorer)
        d08_total   = ('d08',  'sum'),   # total rooms
        d09_total   = ('d09',  'sum'),   # total habitable rooms
        d10_total   = ('d10',  'sum'),   # total sleeping rooms
        d14_worst   = ('d14',  'max'),   # worst floor material code
        d15_worst   = ('d15',  'max'),   # worst wall material code
        d16_worst   = ('d16',  'max'),   # worst roof material code
        floor_score_min = ('floor_score','min'),
        wall_score_min  = ('wall_score', 'min'),
        roof_score_min  = ('roof_score', 'min'),
        n_units     = ('d03',  'count'),
    )
    .reset_index()
)

# Verify
print(f'Dwelling input rows   : {len(dw):,}')
print(f'Dwelling aggregated   : {len(dw_agg):,} unique HHs')
print(f'HHs with 2+ units     : {(dw_agg.n_units > 1).sum():,}')
print(f'Coverage of 21,347    : {len(dw_agg)/21347*100:.1f}%')

In [ ]:
# ── 2.1b  Individual aggregation ─────────────────────────────────────
# Derive household-level demographic summaries from the individual file.
# 'hhsize' is already on the household file (b02_length / size) but
# we verify it here. Key derived variables:
#   hh_size       — household size
#   hh_n_women    — female members
#   hh_n_under15  — children under 15 (dependency)
#   hh_head_sex   — sex of HH head (b04 where hhid__id == 1)

ind2 = ind.copy()
ind2['b05_years'] = pd.to_numeric(ind2['b05_years'], errors='coerce')
ind2['b04']       = pd.to_numeric(ind2['b04'],       errors='coerce')

ind_agg = (
    ind2.groupby('interview__key')
    .apply(lambda g: pd.Series({
        'hh_size'      : len(g),
        'hh_n_women'   : (g['b04'] == 2).sum(),
        'hh_n_under15' : (g['b05_years'] < 15).sum(),
        'hh_head_sex'  : g.loc[g['hhid__id'] == 1, 'b04'].iloc[0]
                          if (g['hhid__id'] == 1).any() else np.nan,
    }))
    .reset_index()
)

print(f'Individual input rows : {len(ind):,}')
print(f'HHs in ind_agg        : {len(ind_agg):,}')
print(f'Mean HH size          : {ind_agg.hh_size.mean():.2f}')
print(f'HH head female (%)    : {(ind_agg.hh_head_sex == 2).mean()*100:.1f}%')

In [ ]:
# ── 2.1c  Land parcels aggregation ───────────────────────────────────
# land_parcels covers ONLY i00 == 1 households (land owners, ~45.5%).
# For each owning HH, take the WORST (most insecure) parcel values.
# Non-owners are NOT in this file — they receive worst-case imputation
# in the final spine (i05=4 squatting, i06=14 no docs, i12=5 extremely likely eviction).

lp2 = lp.copy()

# i05: tenure system  (1=Freehold best, 4=Squatting worst, 98=DK)
# i06: document type  (1=Title deed best, 14=None worst, 15=DK)
# i12: eviction risk  (1=Not likely best, 5=Extremely likely worst, 98=DK)
# i08: right to sell  (1=Yes, 0=No, 98=DK)
# i10: right to bequeath (1=Yes, 0=No, 98=DK)

lp_agg = (
    lp2.groupby('interview__key')
    .agg(
        lp_n_parcels    = ('land_parcels__id', 'count'),
        lp_i05_worst    = ('i05', lambda x: x[x!=98].max() if (x!=98).any() else np.nan),
        lp_i06_worst    = ('i06', lambda x: x[x.isin([14,15])].count()),  # count no-doc parcels
        lp_i12_worst    = ('i12', lambda x: x[x!=98].max() if (x!=98).any() else np.nan),
        lp_i08_any_sell = ('i08', lambda x: (x==1).any().astype(int)),
        lp_i10_any_beq  = ('i10', lambda x: (x==1).any().astype(int)),
        lp_i01_3_mode   = ('i01_3', lambda x: x.mode()[0] if len(x)>0 else np.nan),
    )
    .reset_index()
)

print(f'Land parcel input rows: {len(lp):,}')
print(f'Owner HHs in lp_agg   : {len(lp_agg):,}')
print(f'Expected (i00==1 HHs) : {(hh["i00"]==1).sum():,}')

In [ ]:
# ── 2.1d  Build master spine ─────────────────────────────────────────
# Join order:
#   household (spine, 21,347)
#   + dwelling aggregates     → left join on interview__key
#   + individual aggregates   → left join on interview__key
#   + land parcel aggregates  → left join on interview__key (non-owners stay NaN)
# Then worst-case impute land parcel columns for non-owners.

spine = hh.copy()

# Join dwelling
spine = spine.merge(dw_agg,  on='interview__key', how='left', suffixes=('', '_dw'))

# Join individual
spine = spine.merge(ind_agg, on='interview__key', how='left', suffixes=('', '_ind'))

# Join land parcels
spine = spine.merge(lp_agg,  on='interview__key', how='left', suffixes=('', '_lp'))

# Worst-case imputation for non-owners (i00 == 0): they have no land, no documents.
non_owners = spine['i00'] == 0
spine.loc[non_owners, 'lp_n_parcels']    = 0
spine.loc[non_owners, 'lp_i05_worst']    = 4    # squatting
spine.loc[non_owners, 'lp_i12_worst']    = 5    # extremely likely eviction
spine.loc[non_owners, 'lp_i08_any_sell'] = 0    # no right to sell
spine.loc[non_owners, 'lp_i10_any_beq']  = 0    # no right to bequeath
spine.loc[non_owners, 'lp_i06_worst']    = 1    # 1 parcel with no docs (worst)
spine.loc[non_owners, 'lp_i01_3_mode']   = 10   # moved in without permission

# Derived: persons per room (overcrowding proxy)
spine['persons_per_room'] = spine['hh_size'] / spine['d09_total'].replace(0, np.nan)

# Decode to readable labels
spine['county_name'] = spine['a01'].map(COUNTY_MAP)
spine['residence']   = spine['a07_1'].map({1: 'Urban', 2: 'Rural'})
spine['land_owner']  = spine['i00'].map({0: 'Non-owner', 1: 'Land owner'})

print('='*55)
print('MASTER SPINE — FINAL SHAPE')
print('='*55)
print(f'  Rows     : {spine.shape[0]:,}  (expected 21,347)')
print(f'  Columns  : {spine.shape[1]:,}')
print(f'  Key cols present: {[c for c in ["interview__key","a01","a07_1","i00","county_name","residence"] if c in spine.columns]}')

# Verify zero row inflation
assert spine.shape[0] == 21347, f'Row count mismatch! Got {spine.shape[0]}'
print('  Row count check : PASS')

# Verify key join columns present
for col in ['d03_worst','d14_worst','d15_worst','d16_worst',
            'hh_size','lp_i05_worst','lp_i12_worst']:
    assert col in spine.columns, f'Missing column: {col}'
print('  Join columns    : PASS')
print()
print(spine[['interview__key','a01','county_name','a07_1','residence',
             'i00','land_owner','hh_size','persons_per_room',
             'k05','l14','l15','e06','e07','d15_worst','lp_i05_worst']].head(4).to_string())

### 2.2 Missing Values & Consistency Audit

In [ ]:
# ── 2.2  Missing values audit ────────────────────────────────────────
hh_vars = [v for v, info in VARIABLE_REGISTRY.items() if info['file'] == 'household']

print(f'  {"Variable":<8}  {"Type":<12}  {"N Missing":>10}  {"% Missing":>11}  Notes')
print('  ' + '-'*75)
for var in hh_vars:
    if var not in spine.columns:
        print(f'  {var:<8}  NOT IN SPINE'); continue
    n_miss = spine[var].isna().sum()
    pct    = n_miss / len(spine) * 100
    vtype  = VARIABLE_REGISTRY[var]['type']
    note   = VARIABLE_REGISTRY[var]['notes'][:50]
    print(f'  {var:<8}  {vtype:<12}  {n_miss:>10,}  {pct:>10.1f}%  {note}')

print()
print('Note: structural missingness is by survey design, not data error.')
print('  k05 missing → household is an owner (not a renter).')
print('  l14/l15 missing → household is a renter (no owned property).')
print('  k02 missing → household owns its dwelling (no tenancy agreement needed).')

In [ ]:
# ── 2.3  Outlier detection (IQR method, continuous variables) ────────
cont_vars = [v for v, info in VARIABLE_REGISTRY.items()
             if info['type'] == 'continuous' and v in spine.columns]

print(f'  {"Var":<8}  {"Label":<38}  {"N":>7}  {"N Outliers":>12}  {"% Outliers":>12}')
print('  ' + '-'*83)
outlier_summary = {}
for var in cont_vars:
    s  = pd.to_numeric(spine[var], errors='coerce').dropna()
    Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
    IQR     = Q3 - Q1
    lo, hi  = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    n_out   = ((s < lo) | (s > hi)).sum()
    pct     = n_out / len(s) * 100
    outlier_summary[var] = {'n': n_out, 'pct': pct, 'lo': lo, 'hi': hi}
    lbl = VARIABLE_REGISTRY[var]['label'][:37]
    print(f'  {var:<8}  {lbl:<38}  {len(s):>7,}  {n_out:>12,}  {pct:>11.1f}%')

print()
print('Decision: outliers RETAINED. They represent real households.')
print('Monetary variables will be log-transformed before parametric tests.')

In [ ]:
# ── 2.4  Consistency checks ──────────────────────────────────────────
errors = []

# County codes
valid_counties = set(range(1, 48))
hh_counties    = set(spine['a01'].dropna().astype(int).unique())
invalid        = hh_counties - valid_counties
result = '✓ PASS' if not invalid else f'✗ FAIL — invalid: {invalid}'
print(f'County codes (1-47)       : {result}  ({len(hh_counties)}/47 present)')
if invalid: errors.append(f'Invalid county codes: {invalid}')

# i00 values
i00_vals = sorted(spine['i00'].dropna().unique())
result = '✓ PASS' if set(i00_vals) <= {0, 1} else f'✗ FAIL — got {i00_vals}'
print(f'i00 values (0 or 1)       : {result}')

# a07_1 values
a07_vals = sorted(spine['a07_1'].dropna().unique())
result = '✓ PASS' if set(a07_vals) <= {1, 2} else f'✗ FAIL — got {a07_vals}'
print(f'a07_1 values (1 or 2)     : {result}')

# k02 — insecurity flag: verify code 0 does not exist
k02_vals = sorted(spine['k02'].dropna().unique())
result = '✓ PASS (no code 0)' if 0 not in k02_vals else '✗ FAIL — code 0 found (wrong)'
print(f'k02 values (no 0)         : {result}  — values: {k02_vals}')

# Rent negative check
neg_rent = (pd.to_numeric(spine['k05'], errors='coerce') < 0).sum()
result = '✓ PASS' if neg_rent == 0 else f'✗ FAIL — {neg_rent} negative rents'
print(f'k05 no negatives          : {result}')

# HH sizes
hh_sizes = spine['hh_size'].dropna()
print(f'hh_size range             : min={hh_sizes.min():.0f}  max={hh_sizes.max():.0f}  mean={hh_sizes.mean():.1f}')
large = (hh_sizes > 20).sum()
print(f'  HHs > 20 members        : {large} (flagged — not removed)')

# i00 × land parcel coverage
owner_count    = (spine['i00'] == 1).sum()
has_lp         = (spine['i00'] == 1) & spine['lp_n_parcels'].notna() & (spine['lp_n_parcels'] > 0)
print(f'i00==1 HHs with parcels   : {has_lp.sum():,} / {owner_count:,}  ({has_lp.sum()/owner_count*100:.1f}%)')
no_lp_owners   = (spine['i00'] == 1) & (spine['lp_n_parcels'] == 0)
print(f'i00==1 HHs without parcels: {no_lp_owners.sum()} (expected ~0)')

print()
if errors:
    print(f'ISSUES FOUND: {len(errors)}')
    for e in errors: print(f'  - {e}')
else:
    print('All consistency checks passed. ✓')

---
## 3. Descriptive Statistics

In [ ]:
# ── 3.1  Full descriptive statistics — continuous variables ──────────
def descriptive_table(df, cols):
    rows = []
    for col in cols:
        if col not in df.columns: continue
        s = pd.to_numeric(df[col], errors='coerce').dropna()
        rows.append({
            'Var'      : col,
            'Label'    : VARIABLE_REGISTRY[col]['label'][:35],
            'N'        : len(s),
            'Mean'     : s.mean(),
            'Median'   : s.median(),
            'Std Dev'  : s.std(),
            'Variance' : s.var(),
            'Min'      : s.min(),
            'Max'      : s.max(),
            'Range'    : s.max() - s.min(),
            'IQR'      : s.quantile(0.75) - s.quantile(0.25),
            'Skewness' : s.skew(),
        })
    return pd.DataFrame(rows).set_index('Var')

desc = descriptive_table(spine, cont_vars)
print('DESCRIPTIVE STATISTICS — CONTINUOUS VARIABLES (KHS 2023/24)')
print('='*80)
print(desc.round(2).to_string())

desc.to_csv(TABS / 'descriptive_statistics.csv')
print('\nSaved: descriptive_statistics.csv')

In [ ]:
# ── 3.2  Frequency tables — categorical and ordinal variables ─────────
cat_vars = [v for v, info in VARIABLE_REGISTRY.items()
            if info['type'] in ('categorical', 'ordinal', 'binary')
            and info['file'] == 'household' and v in spine.columns]

VALUE_LABELS = {
    'c01_1': {1:'Public water co.', 2:'Private water co.', 3:'Borehole (owned)',
              4:'Borehole (community)', 5:'Dug well (protected)', 6:'Dug well (unprotected)',
              7:'Spring (protected)', 8:'Spring (unprotected)', 9:'Rainwater',
              10:'Surface water', 11:'Bottled water', 96:'Other'},
    'c04'  : {1:'Flush-piped sewer', 2:'Flush-septic', 3:'Flush-pit latrine',
              4:'Flush-cesspol', 5:'Flush-unknown', 6:'VIP pit latrine',
              7:'Pit+slab', 8:'Open pit', 9:'Composting', 10:'Biodigester',
              11:'Bucket', 12:'Hanging latrine', 13:'No facility/bush', 96:'Other'},
    'c10'  : {1:'KPLC grid', 2:'Mini grid', 3:'Generator', 4:'Solar system',
              5:'Solar battery/torch', 6:'Wind', 7:'Biogas', 8:'Kerosene lamp',
              9:'Charcoal', 10:'Wood', 11:'Candle', 12:'None', 96:'Other'},
    'c11'  : {1:'KPLC grid', 2:'Mini grid', 3:'Generator', 4:'Solar',
              5:'Wind', 6:'Biogas', 7:'LPG gas', 8:'Ethanol',
              9:'Firewood', 10:'Biomass pellets', 11:'Charcoal',
              12:'Crop residue', 13:'Dung/waste', 14:'Not applicable', 96:'Other'},
    'e06'  : {0:'Not flood-prone', 1:'Severe flooding', 2:'Mild flooding'},
    'e07'  : {0:'Not mudslide-prone', 1:'Severe mudslide', 2:'Mild mudslide'},
    'e08'  : {1:'Plain/Flat', 2:'Slightly slopy', 3:'Slopy', 4:'Steep'},
    'i00'  : {0:'Does NOT own land', 1:'Owns land'},
    'k02'  : {1:'Written agreement (secure)', 2:'No written agreement (INSECURE)'},
    'a07_1': {1:'Urban', 2:'Rural'},
    'a01'  : COUNTY_MAP,
}

print('FREQUENCY TABLES — CATEGORICAL / ORDINAL / BINARY VARIABLES')
print()
for var in cat_vars:
    lbl  = VARIABLE_REGISTRY[var]['label']
    vc   = spine[var].value_counts(dropna=False).sort_index()
    labs = VALUE_LABELS.get(var, {})
    print(f'  {var} — {lbl}')
    print(f'  {"Value":>8}  {"Label":<35}  {"Count":>8}  {"% "}')
    for val, cnt in vc.items():
        val_lbl = labs.get(val, '') if not pd.isna(val) else '(missing)'
        pct     = cnt / len(spine) * 100
        bar     = '█' * int(pct / 2)
        print(f'  {str(val):>8}  {val_lbl:<35}  {cnt:>8,}  {pct:>5.1f}%  {bar}')
    print()

In [ ]:
# ── 3.3  Cross-tabulation: Urban/Rural × Land Ownership ──────────────
ct = pd.crosstab(
    spine['a07_1'].map({1: 'Urban', 2: 'Rural'}),
    spine['i00'].map({0: 'Non-owner', 1: 'Land owner'}),
    margins=True
)
print('CROSS-TAB: Urban/Rural × Land Ownership')
print(ct)
print('\nRow percentages:')
ct_pct = pd.crosstab(
    spine['a07_1'].map({1: 'Urban', 2: 'Rural'}),
    spine['i00'].map({0: 'Non-owner', 1: 'Land owner'}),
    normalize='index'
) * 100
print(ct_pct.round(1))
ct.to_csv(TABS / 'crosstab_urban_landowner.csv')

print()
# Cross-tab: flood exposure × residence
if 'e06' in spine.columns:
    ct2 = pd.crosstab(
        spine['residence'],
        spine['e06'].map({0:'No flood', 1:'Severe flood', 2:'Mild flood'}),
        margins=True
    )
    print('CROSS-TAB: Residence × Flood Exposure')
    print(ct2)

---
## 4. Graphical Summaries

In [ ]:
# ── 4.1  Histograms: continuous variables ────────────────────────────
n_cv = len(cont_vars)
fig, axes = plt.subplots(1, n_cv, figsize=(6*n_cv, 5))
if n_cv == 1: axes = [axes]

for ax, var in zip(axes, cont_vars):
    data = pd.to_numeric(spine[var], errors='coerce').dropna()
    dw_  = data[data <= data.quantile(0.99)]
    ax.hist(dw_, bins=60, color=TEAL, edgecolor='white', alpha=0.85)
    ax.axvline(dw_.mean(),   color=RED,  ls='--', lw=2, label=f'Mean={dw_.mean():,.0f}')
    ax.axvline(dw_.median(), color=BLUE, ls=':',  lw=2, label=f'Med={dw_.median():,.0f}')
    ax.set_title(VARIABLE_REGISTRY[var]['label'][:35], pad=8)
    ax.set_xlabel('Value (KES)'); ax.set_ylabel('Count'); ax.legend(fontsize=8)
    ax.text(0.98, 0.92, f'Skew={dw_.skew():.2f}\nn={len(dw_):,}',
            transform=ax.transAxes, ha='right', fontsize=8,
            bbox=dict(boxstyle='round', fc='white', alpha=0.7))

plt.suptitle('Histograms — Continuous Housing Variables (KHS 2023/24)',
             fontsize=14, fontweight='700')
plt.tight_layout()
plt.savefig(FIGS/'fig01_histograms.png', dpi=130, bbox_inches='tight')
plt.show(); print('Saved: fig01_histograms.png')

In [ ]:
# ── 4.2  Boxplots: continuous variables by urban/rural ───────────────
n_cv = len(cont_vars)
fig, axes = plt.subplots(1, n_cv, figsize=(6*n_cv, 6))
if n_cv == 1: axes = [axes]

for ax, var in zip(axes, cont_vars):
    df_  = spine[[var, 'residence']].copy()
    df_[var] = pd.to_numeric(df_[var], errors='coerce')
    df_ = df_.dropna()
    p99 = df_[var].quantile(0.99)
    df_ = df_[df_[var] <= p99]
    groups = [df_[df_['residence'] == r][var].values for r in ['Urban', 'Rural']]

    bp = ax.boxplot(groups, patch_artist=True,
                    medianprops=dict(color=DARK, lw=2.5), notch=True, bootstrap=1000)
    for patch, c in zip(bp['boxes'], [BLUE, GREEN]):
        patch.set_facecolor(c); patch.set_alpha(0.7)
    ax.set_xticks([1, 2]); ax.set_xticklabels(['Urban', 'Rural'])
    ax.set_title(VARIABLE_REGISTRY[var]['label'][:35], pad=8)
    ax.set_ylabel('Value (KES)')

    u, p = stats.mannwhitneyu(groups[0], groups[1], alternative='two-sided')
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'n.s.'
    ax.text(0.5, 0.97, f'MWU p={p:.3f} {sig}',
            transform=ax.transAxes, ha='center', va='top', fontsize=9,
            bbox=dict(boxstyle='round', fc='white', alpha=0.7))

plt.suptitle('Boxplots — Continuous Variables by Residence (KHS 2023/24)',
             fontsize=14, fontweight='700')
plt.tight_layout()
plt.savefig(FIGS/'fig02_boxplots_by_residence.png', dpi=130, bbox_inches='tight')
plt.show(); print('Saved: fig02_boxplots_by_residence.png')

In [ ]:
# ── 4.3  Scatterplots: rent vs housing cost estimates ────────────────
pairs = [('k05', 'l15', 'Rent Paid (KES)',         'Imputed Housing Cost (KES)'),
         ('k05', 'l14', 'Rent Paid (KES)',          'Estimated Dwelling Value (KES)')]
pairs = [(a, b, la, lb) for a, b, la, lb in pairs
         if a in spine.columns and b in spine.columns]

if pairs:
    fig, axes = plt.subplots(1, len(pairs), figsize=(8*len(pairs), 6))
    if len(pairs) == 1: axes = [axes]
    for ax, (xv, yv, xl, yl) in zip(axes, pairs):
        df_  = spine[[xv, yv, 'residence']].copy()
        df_[xv] = pd.to_numeric(df_[xv], errors='coerce')
        df_[yv] = pd.to_numeric(df_[yv], errors='coerce')
        df_ = df_.dropna()
        df_ = df_[(df_[xv] <= df_[xv].quantile(0.99)) & (df_[yv] <= df_[yv].quantile(0.99))]
        for res, color in [('Urban', BLUE), ('Rural', GREEN)]:
            sub = df_[df_['residence'] == res]
            ax.scatter(sub[xv], sub[yv], alpha=0.25, s=12, color=color,
                       label=f'{res} n={len(sub):,}')
        r, p = stats.pearsonr(df_[xv], df_[yv])
        ax.text(0.98, 0.05, f'r={r:.3f}  p={p:.2e}', transform=ax.transAxes,
                ha='right', fontsize=9, bbox=dict(boxstyle='round', fc='white', alpha=0.8))
        ax.set_xlabel(xl); ax.set_ylabel(yl)
        ax.set_title(f'{xl[:20]} vs {yl[:20]}', fontsize=11, fontweight='600')
        ax.legend(fontsize=8)
    plt.suptitle('Scatterplots — Housing Cost Relationships (KHS 2023/24)',
                 fontsize=14, fontweight='700')
    plt.tight_layout()
    plt.savefig(FIGS/'fig03_scatterplots.png', dpi=130, bbox_inches='tight')
    plt.show(); print('Saved: fig03_scatterplots.png')

In [ ]:
# ── 4.4  Correlation heatmaps (Pearson + Spearman) ───────────────────
hmap_vars = [v for v in VARIABLE_REGISTRY if v in spine.columns]
hmap_df   = spine[hmap_vars].apply(pd.to_numeric, errors='coerce')
corr_p    = hmap_df.corr('pearson')
corr_s    = hmap_df.corr('spearman')

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, corr, title in [(axes[0], corr_p, 'Pearson'), (axes[1], corr_s, 'Spearman')]:
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, ax=ax, mask=mask, annot=True, fmt='.2f',
                cmap='RdBu_r', center=0, vmin=-1, vmax=1,
                square=True, linewidths=0.5, linecolor='white',
                cbar_kws={'shrink': 0.8})
    ax.set_title(f'{title} Correlation Heatmap', fontsize=12, fontweight='600', pad=10)
    ax.tick_params(axis='x', rotation=45, labelsize=8)
    ax.tick_params(axis='y', labelsize=8)

plt.suptitle('Correlation Matrices — KHS 2023/24', fontsize=14, fontweight='700')
plt.tight_layout()
plt.savefig(FIGS/'fig04_correlation_heatmaps.png', dpi=130, bbox_inches='tight')
plt.show(); print('Saved: fig04_correlation_heatmaps.png')

print('\nTop |Pearson| correlations (>0.15, excl. self):')
cp_flat = corr_p.where(np.tril(np.ones_like(corr_p), k=-1).astype(bool))
top = cp_flat.stack().reset_index()
top.columns = ['Var1', 'Var2', 'r']
print(top[top['r'].abs() > 0.15].sort_values('r', key=abs, ascending=False).head(10).to_string(index=False))

In [ ]:
# ── 4.5  Bar charts: categorical distributions ───────────────────────
plot_vars = [(v, VARIABLE_REGISTRY[v]['label'])
             for v in ['c01_1', 'c04', 'c10', 'i00', 'e06']
             if v in spine.columns]

fig, axes = plt.subplots(1, len(plot_vars), figsize=(5*len(plot_vars), 5))
if len(plot_vars) == 1: axes = [axes]
for ax, (var, lbl) in zip(axes, plot_vars):
    vc   = spine[var].value_counts(dropna=False).sort_index().head(10)
    labs = VALUE_LABELS.get(var, {})
    tick_labels = [str(labs.get(k, k)) for k in vc.index]
    bars = ax.bar(range(len(vc)), vc.values, color=TEAL, edgecolor='white', alpha=0.85)
    ax.set_xticks(range(len(vc)))
    ax.set_xticklabels(tick_labels, rotation=40, ha='right', fontsize=7)
    ax.set_title(lbl[:35], fontsize=10, fontweight='600')
    ax.set_ylabel('Count')
    for bar, v in zip(bars, vc.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                f'{v/len(spine)*100:.1f}%', ha='center', fontsize=7)

plt.suptitle('Categorical Variable Distributions (KHS 2023/24)', fontsize=14, fontweight='700')
plt.tight_layout()
plt.savefig(FIGS/'fig05_categorical_distributions.png', dpi=130, bbox_inches='tight')
plt.show(); print('Saved: fig05_categorical_distributions.png')

In [ ]:
# ── 4.6  County-level household sample sizes ─────────────────────────
cnt_counts = spine['a01'].value_counts().sort_index()
cnt_labels = [COUNTY_MAP.get(int(c), str(c)) for c in cnt_counts.index]

fig, ax = plt.subplots(figsize=(10, 14))
ax.barh(cnt_labels, cnt_counts.values, color=TEAL, edgecolor='white', alpha=0.85)
ax.axvline(cnt_counts.mean(), color=RED, ls='--', lw=1.8,
           label=f'Mean = {cnt_counts.mean():.0f} HHs')
ax.set_xlabel('Number of Households Surveyed')
ax.set_title('Household Sample Size by County — KHS 2023/24',
             fontsize=13, fontweight='700')
ax.legend()
plt.tight_layout()
plt.savefig(FIGS/'fig06_county_sample_sizes.png', dpi=130, bbox_inches='tight')
plt.show(); print('Saved: fig06_county_sample_sizes.png')

---
## 5. Distributional Assessment

Before applying parametric tests, we verify whether continuous variables follow a normal distribution. The Shapiro-Wilk results here are the **critical decision gate** — they determine which test methods are appropriate in the inference notebook.

Because the KHS sample is large (up to 21,347 observations), we draw a stratified random sample of 5,000 for the Shapiro-Wilk test; very large samples detect trivially small deviations from normality.

In [ ]:
# ── 5.1  KDE + Normal overlay ────────────────────────────────────────
n_cv = len(cont_vars)
fig, axes = plt.subplots(1, n_cv, figsize=(6*n_cv, 5))
if n_cv == 1: axes = [axes]

for ax, var in zip(axes, cont_vars):
    data = pd.to_numeric(spine[var], errors='coerce').dropna()
    dw_  = data[data <= data.quantile(0.99)]
    ax.hist(dw_, bins=60, density=True, color=TEAL, edgecolor='white', alpha=0.65, label='Observed')
    dw_.plot.kde(ax=ax, color=RED, lw=2, label='KDE')
    xr = np.linspace(dw_.min(), dw_.max(), 300)
    ax.plot(xr, spnorm.pdf(xr, dw_.mean(), dw_.std()),
            color=DARK, lw=2, ls='--', label='Normal fit')
    ax.set_title(VARIABLE_REGISTRY[var]['label'][:35], fontsize=10, fontweight='600')
    ax.set_xlabel('Value'); ax.set_ylabel('Density'); ax.legend(fontsize=8)

plt.suptitle('Density Plots — KDE vs Normal Fit (KHS 2023/24)', fontsize=14, fontweight='700')
plt.tight_layout()
plt.savefig(FIGS/'fig07_kde_normal.png', dpi=130, bbox_inches='tight')
plt.show(); print('Saved: fig07_kde_normal.png')

In [ ]:
# ── 5.2  Q-Q Plots ───────────────────────────────────────────────────
n_cv = len(cont_vars)
fig, axes = plt.subplots(1, n_cv, figsize=(6*n_cv, 5))
if n_cv == 1: axes = [axes]

for ax, var in zip(axes, cont_vars):
    data = pd.to_numeric(spine[var], errors='coerce').dropna()
    dw_  = data[data <= data.quantile(0.99)]
    probplot(dw_, dist='norm', plot=ax)
    ax.set_title(f'Q-Q: {VARIABLE_REGISTRY[var]["label"][:30]}', fontsize=10, fontweight='600')
    ax.get_lines()[0].set(color=TEAL, alpha=0.6, markersize=3)
    ax.get_lines()[1].set(color=RED, lw=2)

plt.suptitle('Normal Q-Q Plots — Continuous Variables (KHS 2023/24)',
             fontsize=14, fontweight='700')
plt.tight_layout()
plt.savefig(FIGS/'fig08_qq_plots.png', dpi=130, bbox_inches='tight')
plt.show(); print('Saved: fig08_qq_plots.png')

In [ ]:
# ── 5.3  Shapiro-Wilk normality tests  <-- CRITICAL DECISION GATE ────
# H0: variable is normally distributed
# H1: variable is not normally distributed
# Alpha = 0.05
# Sample = min(5000, n) to avoid power inflation on large N.

print('SHAPIRO-WILK NORMALITY TEST RESULTS')
print('='*78)
print(f'  {"Var":<8}  {"Label":<35}  {"W":>10}  {"p-value":>12}  {"Normal?":>9}')
print('  ' + '-'*78)

normality_results = {}
for var in cont_vars:
    data = pd.to_numeric(spine[var], errors='coerce').dropna()
    dw_  = data[data <= data.quantile(0.99)]
    samp = dw_.sample(min(5000, len(dw_)), random_state=42)
    W, p = shapiro(samp)
    is_n = p > 0.05
    normality_results[var] = {'W': W, 'p': p, 'normal': is_n}
    lbl  = VARIABLE_REGISTRY[var]['label'][:34]
    flag = 'YES' if is_n else 'NO'
    print(f'  {var:<8}  {lbl:<35}  {W:>10.4f}  {p:>12.4e}  {flag:>9}')

print()
n_pass = sum(v['normal'] for v in normality_results.values())
n_fail = len(normality_results) - n_pass
print(f'Variables PASSING normality (p > 0.05) : {n_pass}')
print(f'Variables FAILING normality (p <= 0.05): {n_fail}')
print()
print('=> Non-normal → use nonparametric tests (Mann-Whitney U, Kruskal-Wallis, Spearman)')
print('=> Normal     → eligible for parametric tests (t-test, ANOVA, Pearson)')
print('=> Monetary variables failing normality: check log-transformed version below.')

pd.DataFrame(normality_results).T.to_csv(TABS/'shapiro_wilk_results.csv')
print('\nSaved: shapiro_wilk_results.csv')

In [ ]:
# ── 5.4  Log-transformation check ────────────────────────────────────
print('LOG-TRANSFORMATION NORMALITY CHECK')
print('='*68)
print(f'  {"Var":<8}  {"Skew raw":>10}  {"Skew log":>10}  {"SW p raw":>12}  {"SW p log":>12}  {"Better?"}')
print('  ' + '-'*65)

for var in cont_vars:
    raw = pd.to_numeric(spine[var], errors='coerce').dropna()
    raw = raw[raw > 0]
    log = np.log1p(raw)
    sr  = raw.sample(min(5000, len(raw)), random_state=42)
    sl  = np.log1p(sr)
    _, p_r = shapiro(sr)
    _, p_l = shapiro(sl)
    better = 'Yes' if abs(log.skew()) < abs(raw.skew()) else 'No'
    print(f'  {var:<8}  {raw.skew():>10.2f}  {log.skew():>10.2f}  {p_r:>12.3e}  {p_l:>12.3e}  {better}')

print('\nConclusion: log-transformed versions used in parametric tests requiring normality.')

In [ ]:
# ── 5.5  Q-Q raw vs log side by side ─────────────────────────────────
n_cv = len(cont_vars)
fig, axes = plt.subplots(n_cv, 2, figsize=(12, 5*n_cv))
if n_cv == 1: axes = axes.reshape(1, 2)

for i, var in enumerate(cont_vars):
    raw = pd.to_numeric(spine[var], errors='coerce').dropna()
    raw = raw[raw > 0]
    log = np.log1p(raw)
    lbl = VARIABLE_REGISTRY[var]['label'][:28]
    for ax, data, suffix in [(axes[i, 0], raw, '(Raw)'), (axes[i, 1], log, '(log1p)')]:
        samp = data.sample(min(5000, len(data)), random_state=42)
        probplot(samp, dist='norm', plot=ax)
        ax.set_title(f'Q-Q: {lbl} {suffix}', fontsize=10, fontweight='600')
        ax.get_lines()[0].set(color=TEAL, alpha=0.5, markersize=3)
        ax.get_lines()[1].set(color=RED, lw=2)

plt.suptitle('Q-Q Plots: Raw vs Log-Transformed Variables (KHS 2023/24)',
             fontsize=14, fontweight='700')
plt.tight_layout()
plt.savefig(FIGS/'fig09_qq_log_transform.png', dpi=130, bbox_inches='tight')
plt.show(); print('Saved: fig09_qq_log_transform.png')

---
## 6. EDA Summary & Analysis-Ready Export

In [ ]:
# ── 6.1  EDA findings summary ────────────────────────────────────────
n_hh      = len(spine)
n_cnts    = spine['a01'].nunique()
pct_urban = (spine['a07_1'] == 1).mean() * 100
pct_own   = (spine['i00']   == 1).mean() * 100
pct_flood = (pd.to_numeric(spine['e06'], errors='coerce') > 0).mean() * 100

print(f'''
DATASET OVERVIEW
  Households surveyed : {n_hh:,}
  Counties covered    : {n_cnts}/47
  % Urban households  : {pct_urban:.1f}%
  % Land owners       : {pct_own:.1f}%
  % Flood-exposed     : {pct_flood:.1f}%

DISTRIBUTIONAL FINDINGS
  All monetary variables (k05, l14, l15) are strongly right-skewed.
  Shapiro-Wilk rejects normality at alpha=0.05 for all three.
  Log transformation (log1p) substantially reduces skewness.
  => Nonparametric tests are the primary approach.
     Parametric tests applied to log-transformed versions as sensitivity check.

DATA QUALITY NOTES (CONFIRMED)
  k02 insecurity flag  : use k02 == 2 (No agreement). Code 0 does not exist.
  k25 is NOT rent      : k25 = max willing-to-pay ceiling. Use k05 for actual rent.
  j09/j10/j11 excluded : these are regulation opinion questions, not hazard variables.
  D3 hazard variables  : e06 (flood), e07 (mudslide), e08 (terrain) — all confirmed.
  Land parcels         : structural — only i00==1 households (~45.5%). Non-owners
                         received worst-case imputation in lp_* columns.

RESEARCH QUESTIONS FOR PARTS C AND D
  C1. Is mean log-rent significantly different from a reference level?
      (one-sample t-test on log_k05)
  C2. Do urban and rural households pay significantly different rents?
      (two-sample t-test on log_k05 by a07_1)
  C3. Does rent differ across dwelling quality tiers?
      (one-way ANOVA; groups defined by wall material d15_worst)
  D1. Mann-Whitney U: urban vs rural rent (no normality assumed)
  D2. Kruskal-Wallis: rent across dwelling quality tiers
  D3. Spearman correlation: flood exposure (e06) vs housing cost (k05/l15)
  D4. Bootstrap CI: mean rent burden with uncertainty
''')

In [ ]:
# ── 6.2  Save analysis-ready spine ───────────────────────────────────
# Add log-transformed monetary variables for inference notebook.

spine_out = spine.copy()
for var in cont_vars:
    col = pd.to_numeric(spine[var], errors='coerce').clip(lower=0)
    spine_out[f'log_{var}'] = np.log1p(col)

# Convenience binary flags (referenced in EDA synthesis)
spine_out['no_piped_water']  = (~spine_out['c01_1'].isin([1, 2])).astype(int)
spine_out['open_defecation'] = spine_out['c04'].isin([8, 11, 12, 13]).astype(int)
spine_out['no_electricity']  = spine_out['c10'].isin([9, 10, 12]).astype(int)
spine_out['solid_fuel']      = spine_out['c11'].isin([9, 10, 11, 12, 13]).astype(int)
spine_out['overcrowded']     = (spine_out['persons_per_room'] >= 3).astype(int)

out_path = TABS / 'spine_analysis_ready.parquet'
spine_out.to_parquet(out_path, index=False)
print(f'Saved : {out_path}')
print(f'Shape : {spine_out.shape[0]:,} rows × {spine_out.shape[1]:,} cols')
print()
print('Key columns in analysis-ready spine:')
key_cols = (['interview__key', 'a01', 'county_name', 'a07_1', 'residence',
             'i00', 'land_owner', 'hh_size', 'persons_per_room',
             'k05', 'l14', 'l15', 'log_k05', 'log_l14', 'log_l15',
             'e06', 'e07', 'e08',
             'd14_worst', 'd15_worst', 'd16_worst', 'd03_worst',
             'lp_i05_worst', 'lp_i12_worst', 'lp_n_parcels',
             'no_piped_water', 'open_defecation', 'no_electricity',
             'solid_fuel', 'overcrowded'])
present = [c for c in key_cols if c in spine_out.columns]
print(present)

---
## References

Kenya National Bureau of Statistics. (2024). *Kenya Housing Survey 2023/24 Microdata.* https://statistics.knbs.or.ke/nada/index.php/catalog/184/get-microdata

McKinney, W. (2010). Data structures for statistical computing in Python. *Proceedings of the 9th Python in Science Conference*, 51–56.

Virtanen, P., et al. (2020). SciPy 1.0. *Nature Methods*, 17, 261–272.

Waskom, M. (2021). seaborn: Statistical data visualization. *Journal of Open Source Software*, 6(60), 3021.

---
*End of Notebook — Data Loading, Understanding & Exploration*
